# Getting started

This page builds a SWAP model from scratch, runs it and plots the result: one year of a bare soil with three layers (sand over clay over sand), your own daily weather data in a pandas DataFrame, and a groundwater level held at 120 cm below the surface. Every table is given as a DataFrame and is checked against its schema when it is passed to a component.

Install pySWAP first (see [Installation](../installation/)). The SWAP executable is downloaded the first time a model runs.

In [ ]:
import pandas as pd

import pyswap as ps

## Weather data

SWAP needs daily records with the columns `STATION, DD, MM, YYYY, RAD, TMIN, TMAX, HUM, WIND, RAIN, ETREF, WET` (units are listed in the `MetFile` docstring). Any DataFrame with these columns works; here we read a station file that ships with pySWAP. Replace the path with your own file.

In [ ]:
weather = pd.read_csv(ps.testcase.get_path("hupselbrook", "met"))
weather.columns = weather.columns.str.upper()
weather = weather[weather["YYYY"] == 2003]
weather.head()

In [ ]:
meteorology = ps.components.Meteorology(
    metfile=ps.components.MetFile(metfil="site.met", content=weather),
    lat=52.0,  # latitude [degrees]
    alt=21.0,  # altitude [m]
    altw=10.0,  # height of the wind measurement [m]
    swetr=0,  # compute ET with Penman-Monteith from the weather data
    angstroma=0.25,
    angstromb=0.5,
    swdivide=1,  # divide ET into evaporation and transpiration
    swmetdetail=0,  # daily weather data
    swrain=0,  # daily rainfall amounts
    swetsine=0,
)

## Simulation period and output

The model runs for 2003 and writes two CSV files: daily totals of the water balance terms (`inlist_csv`) and the water content and pressure head of every soil compartment (`inlist_csv_tz`).

In [ ]:
generalsettings = ps.components.GeneralSettings(
    tstart="2003-01-01",
    tend="2003-12-31",
    nprintday=1,
    swmonth=0,
    period=1,  # daily output
    swres=0,
    swodat=0,
    swyrvar=0,
    datefix="31 12",
    extensions=["csv", "csv_tz"],
    inlist_csv=["rain", "epot", "eact", "qbottom", "gwl"],
    inlist_csv_tz=["wc", "h"],
)

richardsettings = ps.components.RichardsSettings(
    swkmean=1,
    swkimpl=0,
    dtmin=1e-6,
    dtmax=0.04,
    gwlconv=100.0,
    critdevh1cp=1e-3,
    critdevh2cp=1e-2,
    critdevponddt=1e-4,
    maxit=30,
    maxbacktr=5,
)

## Soil

The profile is 2 m deep: 40 cm of sand, 40 cm of clay and 120 cm of sand. `SOILPROFILE` divides each layer into compartments; `SOILHYDRFUNC` gives the Mualem-van Genuchten parameters of each soil, one row per layer (illustrative values).

In [ ]:
soilprofile = ps.components.SoilProfile(
    swsophy=0,  # analytical soil hydraulic functions
    swhyst=0,  # no hysteresis
    swmacro=0,  # no macropores
    soilprofile=pd.DataFrame({
        "ISUBLAY": [1, 2, 3],
        "ISOILLAY": [1, 2, 3],
        "HSUBLAY": [40.0, 40.0, 120.0],  # layer thickness [cm]
        "HCOMP": [1.0, 2.0, 5.0],  # compartment thickness [cm]
        "NCOMP": [40, 20, 24],
    }),
    soilhydrfunc=pd.DataFrame({  # sand, clay, sand
        "ORES": [0.02, 0.01, 0.02],
        "OSAT": [0.38, 0.50, 0.36],
        "ALFA": [0.028, 0.015, 0.035],
        "NPAR": [1.80, 1.25, 2.10],
        "KSATFIT": [25.0, 2.0, 40.0],  # [cm/d]
        "LEXP": [0.5, -3.0, 0.5],
        "ALFAW": [0.056, 0.030, 0.070],
        "H_ENPR": [0.0, 0.0, 0.0],
        "KSATEXM": [25.0, 2.0, 40.0],
        "BDENS": [1500.0, 1200.0, 1550.0],  # [mg/cm3]
    }),
)

## Boundaries

At the top: bare soil (no crop), no runoff from elsewhere, and soil evaporation reduced as the topsoil dries. There is no lateral drainage. At the bottom, the groundwater level is prescribed (`swbotb=1`) and held at 120 cm below the surface; the model starts in equilibrium with it.

In [ ]:
crop = ps.components.Crop(swcrop=0)
lateraldrainage = ps.components.Drainage(swdra=0)
surfaceflow = ps.components.SurfaceFlow(
    swrunon=0, swpondmx=0, pondmx=0.2, rsro=0.5, rsroexp=1.0
)
evaporation = ps.components.Evaporation(
    swcfbs=0, swredu=2, cofredbo=0.54, rsoil=150.0
)
soilmoisture = ps.components.SoilMoisture(swinco=2, gwli=-120.0)
bottomboundary = ps.components.BottomBoundary(
    swbbcfile=0,
    swbotb=1,
    gwlevel=pd.DataFrame({
        "DATE1": ["2003-01-01", "2003-12-31"],
        "GWLEVEL": [-120.0, -120.0],  # [cm]
    }),
)

## Run

The components are put together in a `Model`. `run()` validates the model, writes the input files, runs SWAP and returns a `Result`.

In [ ]:
ml = ps.Model(
    metadata=ps.components.Metadata(
        author="Your name",
        institution="Your institution",
        email="you@example.org",
        project="getting-started",
        swap_ver="4.2",
    ),
    generalsettings=generalsettings,
    richardsettings=richardsettings,
    meteorology=meteorology,
    crop=crop,
    soilmoisture=soilmoisture,
    surfaceflow=surfaceflow,
    evaporation=evaporation,
    soilprofile=soilprofile,
    lateraldrainage=lateraldrainage,
    bottomboundary=bottomboundary,
)
result = ml.run()

## Results

`result.csv` holds the daily water balance terms [cm]; their sums give the yearly balance (a negative `QBOTTOM` is drainage to the groundwater).

In [ ]:
result.csv[["RAIN", "EPOT", "EACT", "QBOTTOM"]].sum()

`result.csv_tz` holds the water content of every compartment. The heat map shows the rain wetting the top sand layer, the clay staying wet, and the lower sand at the groundwater level.

In [ ]:
ps.plot.water_content(
    result.csv_tz.reset_index(), depth_col="DEPTH", date_col="DATE", wcontent_col="WC"
)

From here, the [tutorials](../../tutorials/) add crops, drainage and other processes, and the [Installation](../installation/) page shows how to run the SWAP testcases.